In [1]:
pip install -v srcml_caller

Using pip 23.1.2 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 4.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import io
import re
import tokenize
import nltk
from pygments import lex
from pygments.lexers import PythonLexer
from pygments.token import Token
from nltk.tokenize import RegexpTokenizer
import xml.etree.ElementTree as ET
from io import BytesIO
import srcml_caller

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
leetcode_df = pd.read_csv('/content/drive/MyDrive/thesis/leetcode data-extraction/leetcode_code_df.csv', index_col= 0)

leetcode_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29732 entries, 0 to 29740
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   username                29732 non-null  object 
 1   country                 29732 non-null  object 
 2   contest_url             29732 non-null  object 
 3   num_of_contest          29732 non-null  int64  
 4   finish_time             29732 non-null  object 
 5   is_weekly               29732 non-null  bool   
 6   rank                    29732 non-null  int64  
 7   score                   29732 non-null  int64  
 8   question_1_language     29708 non-null  object 
 9   question_1_code         18333 non-null  object 
 10  question_1_finish_time  29708 non-null  object 
 11  question_2_language     29551 non-null  object 
 12  question_2_code         18231 non-null  object 
 13  question_2_finish_time  29551 non-null  object 
 14  question_3_language     27475 non-null  obj

In [5]:
leetcode_df.head()

,username,country,contest_url,num_of_contest,finish_time,is_weekly,rank,score,question_1_language,question_1_code,...,question_2_language,question_2_code,question_2_finish_time,question_3_language,question_3_code,question_3_finish_time,question_4_language,question_4_code,question_4_finish_time,user_global_rank
0,fmota,Brazil,https://leetcode.com/contest/weekly-contest-36...,367,00:12:36,True,2,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:04:12,C++,class Solution {\npublic:\n vector<int> fin...,00:06:42,C++,class Solution {\npublic:\n vector<vector<i...,00:12:36,486427.0
1,nicholask_17,Hong Kong,https://leetcode.com/contest/weekly-contest-36...,367,00:13:02,True,3,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:07:49,C++,class Solution {\npublic:\n vector<int> fin...,00:05:19,C++,class Solution {\npublic:\n vector<vector<i...,00:13:02,27684.0
2,skywalkert,China,https://leetcode.com/contest/weekly-contest-36...,367,00:13:24,True,4,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:13:24,C++,class Solution {\npublic:\n vector<int> fin...,00:04:25,C++,class Solution {\npublic:\n vector<vector<i...,00:09:25,16.0
3,hank55663,Taiwan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:31,True,7,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:06:42,C++,class Solution {\npublic:\n vector<int> fin...,00:04:36,C++,class Solution {\npublic:\n const int mod=1...,00:09:31,6234.0
4,DimmyT,Kazakhstan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:44,True,8,17,C++,class Solution {\npublic:\n vector<int> fin...,...,C++,class Solution {\npublic:\n string shortest...,00:06:26,C++,class Solution {\npublic:\n vector<int> fin...,00:09:55,C++,class Solution {\npublic:\n vector<vector<i...,00:14:44,702585.0


In [6]:
leetcode_questions_df = pd.DataFrame([{
                'username': row['username'],
                'country': row['country'],
                'contest_url': row['contest_url'],
                'num_of_contest': row['num_of_contest'],
                'finish_time': row['finish_time'],
                'is_weekly': row['is_weekly'],
                'rank': row['rank'],
                'score': row['score'],
                'user_global_rank': row['user_global_rank'],
                'question_number': i,
                'question_language': row[f'question_{i}_language'],
                'question_code': row[f'question_{i}_code'],
                'question_finish_time': row[f'question_{i}_finish_time']
            } for i in range(1, 5) for index, row in leetcode_df.iterrows()])

leetcode_questions_df.head()

,username,country,contest_url,num_of_contest,finish_time,is_weekly,rank,score,user_global_rank,question_number,question_language,question_code,question_finish_time
0,fmota,Brazil,https://leetcode.com/contest/weekly-contest-36...,367,00:12:36,True,2,17,486427.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:01:58
1,nicholask_17,Hong Kong,https://leetcode.com/contest/weekly-contest-36...,367,00:13:02,True,3,17,27684.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:05:26
2,skywalkert,China,https://leetcode.com/contest/weekly-contest-36...,367,00:13:24,True,4,17,16.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:04:36
3,hank55663,Taiwan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:31,True,7,17,6234.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:04:48
4,DimmyT,Kazakhstan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:44,True,8,17,702585.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:02:06


In [7]:
leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118928 entries, 0 to 118927
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   username              118928 non-null  object 
 1   country               118928 non-null  object 
 2   contest_url           118928 non-null  object 
 3   num_of_contest        118928 non-null  int64  
 4   finish_time           118928 non-null  object 
 5   is_weekly             118928 non-null  bool   
 6   rank                  118928 non-null  int64  
 7   score                 118928 non-null  int64  
 8   user_global_rank      87488 non-null   float64
 9   question_number       118928 non-null  int64  
 10  question_language     104367 non-null  object 
 11  question_code         64598 non-null   object 
 12  question_finish_time  104367 non-null  object 
dtypes: bool(1), float64(1), int64(4), object(7)
memory usage: 11.0+ MB


In [8]:
leetcode_questions_df.loc[leetcode_questions_df['user_global_rank'] == 5000000, 'user_global_rank'] = None

leetcode_questions_df.dropna(inplace=True)

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63654 entries, 0 to 118907
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   username              63654 non-null  object 
 1   country               63654 non-null  object 
 2   contest_url           63654 non-null  object 
 3   num_of_contest        63654 non-null  int64  
 4   finish_time           63654 non-null  object 
 5   is_weekly             63654 non-null  bool   
 6   rank                  63654 non-null  int64  
 7   score                 63654 non-null  int64  
 8   user_global_rank      63654 non-null  float64
 9   question_number       63654 non-null  int64  
 10  question_language     63654 non-null  object 
 11  question_code         63654 non-null  object 
 12  question_finish_time  63654 non-null  object 
dtypes: bool(1), float64(1), int64(4), object(7)
memory usage: 6.4+ MB


In [9]:
leetcode_questions_df.question_language.value_counts()

question_language
C++           39431
python        15241
java           7162
javascript      545
C#              294
go              269
rust            213
kotlin          193
swift            98
ruby             64
php              54
typescript       52
C                22
dart             14
scala             2
Name: count, dtype: int64

In [10]:
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_language'].isin(['C++', 'java', 'python'])]

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 61834 entries, 0 to 118907
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   username              61834 non-null  object 
 1   country               61834 non-null  object 
 2   contest_url           61834 non-null  object 
 3   num_of_contest        61834 non-null  int64  
 4   finish_time           61834 non-null  object 
 5   is_weekly             61834 non-null  bool   
 6   rank                  61834 non-null  int64  
 7   score                 61834 non-null  int64  
 8   user_global_rank      61834 non-null  float64
 9   question_number       61834 non-null  int64  
 10  question_language     61834 non-null  object 
 11  question_code         61834 non-null  object 
 12  question_finish_time  61834 non-null  object 
dtypes: bool(1), float64(1), int64(4), object(7)
memory usage: 6.2+ MB


# Python

In [11]:
def saved_words_count(code, saved_word):

  tokens = list(lex(code, PythonLexer()))

  return sum(1 for token_type, token_value in tokens if token_type is Token.Keyword and token_value == saved_word)

In [12]:
def extract_naming_from_code_python(code):
    names_set = set()
    tokens = list(lex(code, PythonLexer()))

    return {token_value for token_type, token_value in tokens if token_type == Token.Name}

In [13]:
def calculate_comment_density_python(code_string):

    total_lines = 0
    comment_lines = 0

    try:
        byte_stream = BytesIO(code_string.encode('utf-8'))

        for token in tokenize.tokenize(byte_stream.readline):
            if token.type == tokenize.NEWLINE or token.type == tokenize.NL:
                total_lines += 1
            elif token.type == tokenize.COMMENT:
                comment_lines += 1
    except Exception as e:
        print("Exception is",e)

    single_line_comment_density = comment_lines / total_lines if total_lines else 0

    return single_line_comment_density, 0

In [14]:
def comment_tokens_density_python(code_content):
    lines = code_content.split('\n')

    comment_words = 0
    code_words = 0

    for line in lines:
        words = line.split()
        if line.strip().startswith("#"):
            comment_words += len(words)
        else:
            code_words += len(words)

    total_words = comment_words + code_words

    comment_density = comment_words / total_words if total_words > 0 else 0.0

    return comment_density

# C++ And Java

In [15]:
def find_name_tags_from_xml(xml_string):
    root = ET.fromstring(xml_string)

    banned_names = set()
    for type_tag in root.findall('.//type'):
      banned_names.update({name.text for name in type_tag.findall('.//name')})

    return {name.text for name in root.findall('.//name') if name.text not in banned_names}

def find_name_tags_from_code(code_string, CodeLanguage):
    xml = srcml_caller.to_srcml(code_string,
                             CodeLanguage, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    return find_name_tags_from_xml(xml)

In [16]:
def calculate_comment_density_from_xml(xml_string, total_lines):
    root = ET.fromstring(xml_string)

    comment_lines = 0
    multiline_comment_lines = 0

    for comment_element in root.findall('.//comment'):
      if comment_element.attrib['type'] == 'block':
        multiline_comment_lines = comment_element.text.count('\n') + 1
      else :
        comment_lines += 1

    single_line_comment_density = comment_lines / total_lines if total_lines else 0
    multiline_comment_density = multiline_comment_lines / total_lines if total_lines else 0

    return single_line_comment_density, multiline_comment_density


def calculate_comment_density_from_code(code_string, CodeLanguage):
    total_lines = code_string.count('\n')
    xml = srcml_caller.to_srcml(code_string,
                             CodeLanguage, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    return calculate_comment_density_from_xml(xml, total_lines)

# Combine

In [17]:
def extract_naming_from_code(code, language):
    if language == 'python':
        return extract_naming_from_code_python(code)
    elif language == 'C++':
        return find_name_tags_from_code(code, srcml_caller.CodeLanguage.c_plus_plus)
    elif language == 'java':
        return find_name_tags_from_code(code, srcml_caller.CodeLanguage.java)

In [18]:
def count_tokens(code):
  token_pattern = r'[A-Za-z0-9]+'

  tokenizer = RegexpTokenizer(token_pattern)

  tokens = tokenizer.tokenize(code)

  return len(set(tokens))

In [19]:
def calculate_comment_tokens_density_from_xml(xml_string, num_of_tokens):
    root = ET.fromstring(xml_string)

    tokens = set()
    token_pattern = r'[A-Za-z0-9]+'
    tokenizer = RegexpTokenizer(token_pattern)

    for comment_element in root.findall('.//comment'):
      tokens.update(tokenizer.tokenize(comment_element.text))

    return len(set(tokens)) / num_of_tokens

def calculate_comment_tokens_density_from_code(code_string, CodeLanguage):
    xml = srcml_caller.to_srcml(code_string,
                             CodeLanguage, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    return calculate_comment_tokens_density_from_xml(xml, count_tokens(code_string))

In [20]:
def calculate_comment_tokens_density(code, language):
  if language == 'python':
    return comment_tokens_density_python(code)
  elif language == 'C++':
    return calculate_comment_tokens_density_from_code(code, srcml_caller.CodeLanguage.c_plus_plus)
  elif language == 'java':
    return calculate_comment_tokens_density_from_code(code, srcml_caller.CodeLanguage.java)

In [21]:
def number_of_lines_count(code):
  return code.count('\n') + 1

In [22]:
def calculate_function_count(code, language):
  if language == 'python':
    return saved_words_count(code, "def")
  elif language == 'C++':
    xml = srcml_caller.to_srcml(code, srcml_caller.CodeLanguage.c_plus_plus, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    root = ET.fromstring(xml)
    return len(root.findall('.//function'))
  elif language == 'java':
    xml = srcml_caller.to_srcml(code, srcml_caller.CodeLanguage.java, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    root = ET.fromstring(xml)
    return len(root.findall('.//function'))

In [23]:
def calculate_loop_count(code, language):
  if language == 'python':
    return saved_words_count(code, "for")
  elif language == 'C++':
    xml = srcml_caller.to_srcml(code, srcml_caller.CodeLanguage.c_plus_plus, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    root = ET.fromstring(xml)
    return len(root.findall('.//for'))
  elif language == 'java':
    xml = srcml_caller.to_srcml(code, srcml_caller.CodeLanguage.java, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    root = ET.fromstring(xml)
    return len(root.findall('.//for'))

In [24]:
def calculate_condition_count(code, language):
  if language == 'python':
    return saved_words_count(code, "if") + saved_words_count(code, "while")
  elif language == 'C++':
    xml = srcml_caller.to_srcml(code, srcml_caller.CodeLanguage.c_plus_plus, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    root = ET.fromstring(xml)
    return len(root.findall('.//if_stmt')) + len(root.findall('.//while'))
  elif language == 'java':
    xml = srcml_caller.to_srcml(code, srcml_caller.CodeLanguage.java, include_positions = False).replace('xmlns="http://www.srcML.org/srcML/src"', '')
    root = ET.fromstring(xml)
    return len(root.findall('.//if_stmt')) + len(root.findall('.//while'))

In [25]:
def calculate_comment_density(code, language):
  if language == 'python':
    return calculate_comment_density_python(code)
  elif language == 'C++':
    return calculate_comment_density_from_code(code, srcml_caller.CodeLanguage.c_plus_plus)
  elif language == 'java':
    return calculate_comment_density_from_code(code, srcml_caller.CodeLanguage.java)

In [26]:
def extract_features(row):

    print("Starting extracting features for username = {}, rank = {}, question_number = {}, url = {}".format(row['username'], row['rank'], row['question_number'], row['contest_url']))

    code, language = row.question_code, row.question_language

    number_of_lines = number_of_lines_count(code)
    names_set = extract_naming_from_code(code, language)

    token_count = count_tokens(code)

    variables_count = len(names_set)
    function_count = calculate_function_count(code, language)
    loop_count = calculate_loop_count(code, language)
    condition_count = calculate_condition_count(code, language)

    function_density = function_count / token_count if token_count > 0 else 0
    loop_density = loop_count / token_count if token_count > 0 else 0
    condition_density = condition_count / token_count if token_count > 0 else 0
    single_line_comment_density, multiline_comment_density = calculate_comment_density(code, language)
    comment_tokens_density = calculate_comment_tokens_density(code, language)

    features = [number_of_lines, names_set, token_count,
                variables_count, function_count, loop_count, condition_count,
                single_line_comment_density, multiline_comment_density,
                function_density, loop_density, condition_density, comment_tokens_density]

    print("Finished extracting features for username = {}, rank = {}, question_number = {}, url = {}".format(row['username'], row['rank'], row['question_number'], row['contest_url']))
    return features

In [27]:
leetcode_questions_df.head()

,username,country,contest_url,num_of_contest,finish_time,is_weekly,rank,score,user_global_rank,question_number,question_language,question_code,question_finish_time
0,fmota,Brazil,https://leetcode.com/contest/weekly-contest-36...,367,00:12:36,True,2,17,486427.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:01:58
1,nicholask_17,Hong Kong,https://leetcode.com/contest/weekly-contest-36...,367,00:13:02,True,3,17,27684.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:05:26
2,skywalkert,China,https://leetcode.com/contest/weekly-contest-36...,367,00:13:24,True,4,17,16.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:04:36
3,hank55663,Taiwan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:31,True,7,17,6234.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:04:48
4,DimmyT,Kazakhstan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:44,True,8,17,702585.0,1,C++,class Solution {\npublic:\n vector<int> fin...,00:02:06


In [28]:
leetcode_questions_df[['number_of_lines', 'names_set', 'token_count',
                'variables_count', 'function_count', 'loop_count', 'condition_count',
                'single_line_comment_density', 'multiline_comment_density',
                'function_density', 'loop_density', 'condition_density', 'comment_tokens_density']] = leetcode_questions_df.apply(extract_features, axis = 1).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
Starting extracting features for username = abhishekk_1912, rank = 351, question_number = 4, url = https://leetcode.com/contest/biweekly-contest-121/ranking/15
Finished extracting features for username = abhishekk_1912, rank = 351, question_number = 4, url = https://leetcode.com/contest/biweekly-contest-121/ranking/15
Starting extracting features for username = darkaadityaa, rank = 352, question_number = 4, url = https://leetcode.com/contest/biweekly-contest-121/ranking/15
Finished extracting features for username = darkaadityaa, rank = 352, question_number = 4, url = https://leetcode.com/contest/biweekly-contest-121/ranking/15
Starting extracting features for username = just_hands13, rank = 353, question_number = 4, url = https://leetcode.com/contest/biweekly-contest-121/ranking/15
Finished extracting features for username = just_hands13, rank = 353, question_number = 4, url = https://leetcode.com/contest/biweekly-contest-121/ranking/

In [29]:
leetcode_questions_df.head()

,username,country,contest_url,num_of_contest,finish_time,is_weekly,rank,score,user_global_rank,question_number,...,variables_count,function_count,loop_count,condition_count,single_line_comment_density,multiline_comment_density,function_density,loop_density,condition_density,comment_tokens_density
0,fmota,Brazil,https://leetcode.com/contest/weekly-contest-36...,367,00:12:36,True,2,17,486427.0,1,...,12,1,2,1,0.000000,0.0,0.045455,0.090909,0.045455,0.0
1,nicholask_17,Hong Kong,https://leetcode.com/contest/weekly-contest-36...,367,00:13:02,True,3,17,27684.0,1,...,18,1,2,6,0.000000,0.0,0.032258,0.064516,0.193548,0.0
2,skywalkert,China,https://leetcode.com/contest/weekly-contest-36...,367,00:13:24,True,4,17,16.0,1,...,17,1,1,2,0.000000,0.0,0.037037,0.037037,0.074074,0.0
3,hank55663,Taiwan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:31,True,7,17,6234.0,1,...,12,1,1,5,0.037037,0.0,0.040000,0.040000,0.200000,0.2
4,DimmyT,Kazakhstan,https://leetcode.com/contest/weekly-contest-36...,367,00:14:44,True,8,17,702585.0,1,...,10,1,2,1,0.000000,0.0,0.052632,0.105263,0.052632,0.0


In [30]:
leetcode_questions_df.to_csv('/content/drive/MyDrive/thesis/leetcode data-extraction/code_features_extracts.csv', index=False)